In [1]:
# ============================================================
# HIGGS BOSON CLASSIFICATION
# FINAL XGBOOST TUNING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
import joblib

from xgboost import XGBClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from preprocessing_22 import load_data_unscaled


# ============================================================
# LOAD DATA
# ============================================================

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    feature_names
) = load_data_unscaled()

print("Train Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Test Shape:", X_test.shape)


# ============================================================
# BASELINE MODEL
# ============================================================

print("\n" + "=" * 60)
print("BASELINE XGBOOST")
print("=" * 60)

xgb_saved = joblib.load(
    "../models/higgs_xgboost.pkl"
)

print(xgb_saved)


# ============================================================
# Evaluate Saved Model
# ============================================================

y_prob = xgb_saved.predict_proba(X_test)[:,1]

baseline_auc = roc_auc_score(
    y_test,
    y_prob
)

print(
    "Baseline ROC-AUC:",
    round(baseline_auc, 4)
)


# ============================================================
# EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("EARLY STOPPING")
print("=" * 60)

early_stop_model = XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.93,
    reg_alpha=0.1,
    reg_lambda=2.0,
    eval_metric="auc",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

early_stop_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

best_n = early_stop_model.best_iteration

print(f"Best Iteration: {best_n}")


# ============================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================

param_dist = {

    "n_estimators": [best_n],

    "learning_rate": [
        0.01,
        0.03,
        0.05
    ],

    "max_depth": [
        4,
        5,
        6,
        7
    ],

    "min_child_weight": [
        1,
        3,
        5
    ],

    "subsample": [
        0.7,
        0.8,
        0.9
    ],

    "colsample_bytree": [
        0.7,
        0.8,
        0.9
    ],

    "gamma": [
        0,
        0.05,
        0.1
    ],

    "reg_alpha": [
        0,
        0.05,
        0.1,
        0.5
    ],

    "reg_lambda": [
        1,
        2,
        3,
        5
    ],

    "scale_pos_weight": [
        1.5,
        1.93,
        2.5
    ]
}


# ============================================================
# RANDOMIZED SEARCH
# ============================================================

print("\n" + "=" * 60)
print("RANDOMIZED SEARCH")
print("=" * 60)

base_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("\nBest Parameters:")
print(search.best_params_)

print("\nBest CV ROC-AUC:")
print(search.best_score_)


# ============================================================
# FINAL MODEL
# ============================================================

print("\n" + "=" * 60)
print("FINAL TUNED MODEL")
print("=" * 60)

best_params = search.best_params_.copy()

best_params["n_estimators"] = best_n + 200

final_model = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)


# ============================================================
# TEST EVALUATION
# ============================================================

y_pred_final = final_model.predict(X_test)
y_prob_final = final_model.predict_proba(X_test)[:, 1]

final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)
final_auc = roc_auc_score(y_test, y_prob_final)

print(f"Accuracy : {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall   : {final_recall:.4f}")
print(f"F1 Score : {final_f1:.4f}")
print(f"ROC AUC  : {final_auc:.4f}")


# ============================================================
# OVERFITTING CHECK
# ============================================================

print("\n" + "=" * 60)
print("OVERFITTING ANALYSIS")
print("=" * 60)

train_prob = final_model.predict_proba(X_train)[:, 1]
val_prob = final_model.predict_proba(X_val)[:, 1]
test_prob = final_model.predict_proba(X_test)[:, 1]

print(
    "Train AUC      :",
    round(roc_auc_score(y_train, train_prob), 4)
)

print(
    "Validation AUC :",
    round(roc_auc_score(y_val, val_prob), 4)
)

print(
    "Test AUC       :",
    round(roc_auc_score(y_test, test_prob), 4)
)


# ============================================================
# BASELINE VS TUNED
# ============================================================

print("\n" + "=" * 60)
print("BASELINE VS TUNED")
print("=" * 60)

print(f"Baseline ROC-AUC : {baseline_auc:.4f}")
print(f"Tuned ROC-AUC    : {final_auc:.4f}")
print(f"Improvement      : {final_auc - baseline_auc:.4f}")


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": final_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 15 Features")

print(
    importance.head(15)
)
# Confusion Matrix
import seaborn as sns
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("tuned XGBoost Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

plt.savefig(
    "../results/plots/tuned_xgb_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

""""
# ============================================================
# SAVE MODEL
# ============================================================

joblib.dump(
    final_model,
    "../models/higgs_xgboost_tuned.pkl"
)

#final_model.save_model(
 #   "../models/higgs_xgboost_tuned.ubj"
#)

#print("\nModel Saved Successfully")"""

Train Shape: (572766, 30)
Validation Shape: (122736, 30)
Test Shape: (122736, 30)

BASELINE XGBOOST
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)
Baseline ROC-AUC: 0.9105

EARLY STOPPING
Best Iteration: 2035

RANDOMIZED SEARCH
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Be

NameError: name 'confusion_matrix' is not defined

In [3]:
import gzip
import shutil
with gzip.open('../data/atlas-higgs-challenge-2014-v2.csv.gz', 'rb') as f_in:
    with open('../data/atlas-higgs-challenge-2014-v2.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)


In [ ]:
# ============================================================
# 1. IMPORTS & DATA LOADING
# ============================================================
import importlib


import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from preprocessing_22 import load_data_unscaled

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,  
    y_test,
    feature_names
) = load_data_unscaled()


# ============================================================
# 2. BASELINE MODEL (your original — fixed syntax)
# ============================================================
baseline = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.93,
    reg_alpha=0.1,
    reg_lambda=2.0,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)
baseline.fit(X_train, y_train)

y_prob_base = baseline.predict_proba(X_test)[:, 1]
y_pred_base = baseline.predict(X_test)

print("=" * 45)
print("BASELINE RESULTS")
print("=" * 45)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_base):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_base):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_prob_base):.4f}")


# ============================================================
# 3. EARLY STOPPING — find optimal n_estimators
#    Prevents overfitting from training all 2000 trees blindly
# ============================================================
early_stop_model = XGBClassifier(
    n_estimators=3000,          # set high; early stopping will cut it
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.93,
    reg_alpha=0.1,
    reg_lambda=2.0,
    eval_metric="auc",
    early_stopping_rounds=50,   # stop if AUC doesn't improve for 50 rounds
    random_state=42,
    n_jobs=-1
)
early_stop_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

best_n = early_stop_model.best_iteration
print(f"\nBest n_estimators from early stopping: {best_n}")


# ============================================================
# 4. RANDOMIZED HYPERPARAMETER SEARCH
#    Searches across depth, regularization, sampling, eta
# ============================================================
param_dist = {
    "n_estimators"     : [best_n],          # use early-stopped optimum
    "learning_rate"    : [0.01, 0.03, 0.05, 0.08],
    "max_depth"        : [4, 5, 6, 7, 8],
    "min_child_weight" : [1, 3, 5, 7],
    "subsample"        : [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree" : [0.6, 0.7, 0.8, 0.9],
    "colsample_bylevel": [0.6, 0.8, 1.0],
    "reg_alpha"        : [0.0, 0.01, 0.1, 0.5, 1.0],
    "reg_lambda"       : [1.0, 1.5, 2.0, 3.0, 5.0],
    "gamma"            : [0.0, 0.1, 0.3, 0.5],
    "scale_pos_weight" : [1.0, 1.5, 1.93, 2.5],
}

base_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",         # explicit — faster and reproducible
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=50,                  # number of random combinations to try
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)
search.fit(X_train, y_train)

print("\nBest params found:")
for k, v in search.best_params_.items():
    print(f"  {k:22s}: {v}")
print(f"Best CV AUC: {search.best_score_:.4f}")


# ============================================================
# 5. FINAL MODEL — retrain on best params with early stopping
# ============================================================
best_params = search.best_params_.copy()
best_params["n_estimators"] = 3000      # let early stopping decide again

final_model = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

y_prob_final = final_model.predict_proba(X_test)[:, 1]
y_pred_final = final_model.predict(X_test)

print("\n" + "=" * 45)
print("FINAL TUNED MODEL RESULTS")
print("=" * 45)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_final):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_final):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_prob_final):.4f}")


# ============================================================
# 6. BEFORE / AFTER COMPARISON
# ============================================================
print("\n" + "=" * 45)
print("COMPARISON: BASELINE vs TUNED")
print("=" * 45)
metrics = {
    "Accuracy" : (accuracy_score, y_pred_base,  y_pred_final),
    "Precision": (precision_score, y_pred_base, y_pred_final),
    "Recall"   : (recall_score,   y_pred_base,  y_pred_final),
    "F1 Score" : (f1_score,       y_pred_base,  y_pred_final),
}
for name, (fn, base_p, tune_p) in metrics.items():
    b = fn(y_test, base_p)
    t = fn(y_test, tune_p)
    delta = t - b
    arrow = "▲" if delta > 0 else "▼"
    print(f"{name:10s}  Baseline: {b:.4f}  Tuned: {t:.4f}  {arrow} {abs(delta):.4f}")

b_auc = roc_auc_score(y_test, y_prob_base)
t_auc = roc_auc_score(y_test, y_prob_final)
delta_auc = t_auc - b_auc
arrow = "▲" if delta_auc > 0 else "▼"
print(f"{'AUC':10s}  Baseline: {b_auc:.4f}  Tuned: {t_auc:.4f}  {arrow} {abs(delta_auc):.4f}")


# ============================================================
# 7. FEATURE IMPORTANCE
# ============================================================
importance = pd.DataFrame({
    "Feature"   : feature_names,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nTop 15 Features (Tuned Model):")
print(importance.head(15).to_string(index=False))


# ============================================================
# 8. SAVE TUNED MODEL (native format — version-safe)
# ============================================================
#final_model.save_model("higgs_xgboost_tuned_v2.ubj")
#print("\nModel saved to higgs_xgboost_tuned.ubj")

BASELINE RESULTS
Accuracy  : 0.8314
Precision : 0.7186
Recall    : 0.8325
F1 Score  : 0.7714
AUC       : 0.9140
[0]	validation_0-auc:0.86838
[100]	validation_0-auc:0.90081
[200]	validation_0-auc:0.90658
[300]	validation_0-auc:0.90890
[400]	validation_0-auc:0.91020
[500]	validation_0-auc:0.91123
[600]	validation_0-auc:0.91193
[700]	validation_0-auc:0.91240
[800]	validation_0-auc:0.91275
[900]	validation_0-auc:0.91300
[1000]	validation_0-auc:0.91324
[1100]	validation_0-auc:0.91342
[1200]	validation_0-auc:0.91355
[1300]	validation_0-auc:0.91365
[1400]	validation_0-auc:0.91376
[1500]	validation_0-auc:0.91385
[1600]	validation_0-auc:0.91387
[1700]	validation_0-auc:0.91391
[1800]	validation_0-auc:0.91395


KeyboardInterrupt: 

In [ ]:
# ============================================================
# SHAP + LIME EXPLAINABILITY — higgs_xgboost_tuned.pkl
# Requirements: xgboost, shap, lime, scikit-learn,
#               pandas, numpy, matplotlib
# pip install xgboost shap lime scikit-learn
# ============================================================

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
# ============================================================
# Load Tuned XGBoost Model
# ============================================================

xgb = joblib.load(
    "../models/higgs_xgboost_tuned.pkl"
)

print("Model loaded successfully.")

# ============================================================
# Load Dataset
# ============================================================

from preprocessing_22 import load_data_unscaled

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    feature_names
) = load_data_unscaled()

print(X_test.shape)
# Convert to DataFrame so SHAP/LIME receive named columns
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)


# ============================================================
# SECTION A — SHAP
# ============================================================
import shap

# TreeExplainer is the fast exact method for XGBoost/LGBM/RF
# No sampling needed — computes exact Shapley values in O(TLD²)
explainer_shap = shap.TreeExplainer(xgb)

# Compute SHAP values for the full test set
# shap_values shape: (n_samples, n_features)
shap_values = explainer_shap.shap_values(X_test_df)

# ── A1. Global — Bar plot (mean |SHAP|) ─────────────────────
plt.figure()
shap.summary_plot(
    shap_values,
    X_test_df,
    plot_type="bar",
    feature_names=feature_names,
    show=False
)
plt.title("SHAP — Global Feature Importance (mean |SHAP value|)")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_global_bar.png", dpi=150)
plt.close()
print("Saved: shap_global_bar.png")

# ── A2. Global — Beeswarm / Summary plot ────────────────────
# Each dot = one sample; colour = feature value; x-axis = impact on output
plt.figure()
shap.summary_plot(
    shap_values,
    X_test_df,
    feature_names=feature_names,
    show=False
)
plt.title("SHAP — Beeswarm Summary Plot")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_beeswarm.png", dpi=150)
plt.close()
print("Saved: shap_beeswarm.png")

# ── A3. Global — SHAP correlation heatmap (top 10 features) ─
top10_idx   = np.argsort(np.abs(shap_values).mean(axis=0))[::-1][:10]
top10_names = [feature_names[i] for i in top10_idx]

shap_df = pd.DataFrame(shap_values[:, top10_idx], columns=top10_names)
plt.figure(figsize=(10, 8))
plt.imshow(shap_df.corr(), cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(top10_names)), top10_names, rotation=45, ha="right")
plt.yticks(range(len(top10_names)), top10_names)
plt.title("SHAP Value Correlation — Top 10 Features")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_correlation_heatmap.png", dpi=150)
plt.close()
print("Saved: shap_correlation_heatmap.png")

# ── A4. Local — Waterfall plot for one signal sample ────────
# Pick the first test sample predicted as signal (class 1)
signal_idx = int(np.where(model.predict(X_test_df) == 1)[0][0])

shap.waterfall_plot(
    shap.Explanation(
        values        = shap_values[signal_idx],
        base_values   = explainer_shap.expected_value,
        data          = X_test_df.iloc[signal_idx].values,
        feature_names = feature_names
    ),
    show=False
)
plt.title(f"SHAP — Local Waterfall (signal sample #{signal_idx})")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_waterfall_signal.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: shap_waterfall_signal.png  (sample index {signal_idx})")

# ── A5. Local — Force plot for one background sample ────────
background_idx = int(np.where(model.predict(X_test_df) == 0)[0][0])

shap.waterfall_plot(
    shap.Explanation(
        values        = shap_values[background_idx],
        base_values   = explainer_shap.expected_value,
        data          = X_test_df.iloc[background_idx].values,
        feature_names = feature_names
    ),
    show=False
)
plt.title(f"SHAP — Local Waterfall (background sample #{background_idx})")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_waterfall_background.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: shap_waterfall_background.png  (sample index {background_idx})")

# ── A6. Dependence plot — top feature vs. top interaction ───
top1 = top10_names[0]   # highest mean |SHAP|
top2 = top10_names[1]   # used as colour (interaction)

plt.figure()
shap.dependence_plot(
    top1,
    shap_values,
    X_test_df,
    interaction_index=top2,
    feature_names=feature_names,
    show=False
)
plt.title(f"SHAP Dependence — {top1} (coloured by {top2})")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_dependence.png", dpi=150)
plt.close()
print(f"Saved: shap_dependence.png")

# ── A7. Print top-10 SHAP feature ranking ───────────────────
mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names
).sort_values(ascending=False)

print("\n=== SHAP Global Feature Ranking (mean |SHAP|) ===")
print(mean_abs_shap.head(10).to_string())


# ============================================================
# SECTION B — LIME
# ============================================================
from lime import lime_tabular

# LimeTabularExplainer needs numpy arrays, not DataFrames
explainer_lime = lime_tabular.LimeTabularExplainer(
    training_data   = X_train_df.values,
    feature_names   = feature_names,
    class_names     = ["Background", "Signal"],
    mode            = "classification",
    discretize_continuous = True,         # bins continuous features
    random_state    = 42
)

# Wrapper: LIME needs a predict_proba function
def predict_proba(X):
    return model.predict_proba(X)

# ── B1. Local — LIME explanation for one signal sample ──────
lime_exp_signal = explainer_lime.explain_instance(
    data_row        = X_test_df.iloc[signal_idx].values,
    predict_fn      = predict_proba,
    num_features    = 15,                 # top-15 features
    num_samples     = 5000                # perturbation samples
)

fig = lime_exp_signal.as_pyplot_figure(label=1)
plt.title(f"LIME — Local Explanation (signal sample #{signal_idx})")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/lime_local_signal.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: lime_local_signal.png")

# ── B2. Local — LIME explanation for one background sample ──
lime_exp_background = explainer_lime.explain_instance(
    data_row        = X_test_df.iloc[background_idx].values,
    predict_fn      = predict_proba,
    num_features    = 15,
    num_samples     = 5000
)

fig = lime_exp_background.as_pyplot_figure(label=0)
plt.title(f"LIME — Local Explanation (background sample #{background_idx})")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/lime_local_background.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: lime_local_background.png")

# ── B3. Global — Aggregate LIME over N random samples ───────
# Average absolute LIME weights across N samples
# → gives a global importance estimate from local explanations

N_LIME_SAMPLES = 200   # increase for more stable estimates (slower)
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(X_test_df), size=N_LIME_SAMPLES, replace=False)

lime_weights = np.zeros((N_LIME_SAMPLES, len(feature_names)))

print(f"\nRunning LIME on {N_LIME_SAMPLES} samples for global estimate...")
for i, idx in enumerate(sample_indices):
    exp = explainer_lime.explain_instance(
        data_row   = X_test_df.iloc[idx].values,
        predict_fn = predict_proba,
        num_features = len(feature_names),   # all features
        num_samples  = 1000
    )
    for feat, weight in exp.as_list(label=1):
        # LIME feature strings look like "DER_mass_MMC <= 100.5"
        # Extract the feature name from the condition string
        for j, fname in enumerate(feature_names):
            if fname in feat:
                lime_weights[i, j] += abs(weight)
                break
    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{N_LIME_SAMPLES}] done")

mean_lime_weights = pd.Series(
    lime_weights.mean(axis=0),
    index=feature_names
).sort_values(ascending=False)

# Plot aggregated LIME global importance
plt.figure(figsize=(10, 7))
mean_lime_weights.head(15).sort_values().plot(
    kind="barh", color="steelblue", edgecolor="black"
)
plt.xlabel("Mean |LIME weight|")
plt.title(f"LIME — Global Feature Importance (avg over {N_LIME_SAMPLES} samples)")
plt.tight_layout()
plt.savefig("..results/shap_and_lime/lime_global_importance.png", dpi=150)
plt.close()
print("Saved: lime_global_importance.png")

print("\n=== LIME Global Feature Ranking (mean |weight|) ===")
print(mean_lime_weights.head(10).to_string())


# ============================================================
# SECTION C — SHAP vs LIME COMPARISON PLOT
# ============================================================
# Rank features by SHAP, then show LIME rank side by side

shap_rank = mean_abs_shap.rank(ascending=False)
lime_rank  = mean_lime_weights.rank(ascending=False)

comparison_df = pd.DataFrame({
    "SHAP rank" : shap_rank,
    "LIME rank" : lime_rank
}).sort_values("SHAP rank").head(15)

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(len(comparison_df))
width = 0.4

ax.bar(x - width/2, comparison_df["SHAP rank"], width,
       label="SHAP rank", color="coral",   edgecolor="black")
ax.bar(x + width/2, comparison_df["LIME rank"], width,
       label="LIME rank", color="steelblue", edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels(comparison_df.index, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Rank (lower = more important)")
ax.set_title("SHAP vs LIME — Feature Importance Rank Comparison (top 15 by SHAP)")
ax.legend()
ax.invert_yaxis()   # rank 1 at top
plt.tight_layout()
plt.savefig("..results/shap_and_lime/shap_vs_lime_comparison.png", dpi=150)
plt.close()
print("\nSaved: shap_vs_lime_comparison.png")

print("\n=== ALL DONE ===")
print("Outputs:")
print("  shap_global_bar.png")
print("  shap_beeswarm.png")
print("  shap_correlation_heatmap.png")
print("  shap_waterfall_signal.png")
print("  shap_waterfall_background.png")
print("  shap_dependence.png")
print("  lime_local_signal.png")
print("  lime_local_background.png")
print("  lime_global_importance.png")
print("  shap_vs_lime_comparison.png")

Model loaded successfully.
(122736, 30)
